In [0]:

from pyspark.sql import functions as F

df = spark.table("retailnew.bronze.pos")

# Cast types
df = (
    df.withColumn("qty", F.col("qty").cast("int"))
      .withColumn("amount", F.col("amount").cast("double"))
      .withColumn("transaction_date", F.to_timestamp("transaction_date"))
)

# Data quality rules
df = df.filter(F.col("transaction_id").isNotNull())
df = df.filter(F.col("customer_id").isNotNull())
df = df.filter(F.col("amount") > 0)
df = df.filter(F.col("qty") >= 0)

# Deduplicate
df = df.dropDuplicates(["transaction_id"])

# Add metadata
df = df.withColumn("source_file", F.lit(''))

# Write to Silver
(
    df.write
      .format("delta")
      .mode("overwrite")
      .option("overwriteSchema", "true")
      .saveAsTable("retailnew.silver.pos_clean")
)

